Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by giving them access to external,
up-to-date, and relevant information. Instead of relying solely on the knowledge encoded during their training,
RAG models can retrieve information from a separate knowledge base (like a database, documents, or the internet) and then use this retrieved context
to generate more accurate, relevant, and grounded responses.

In [ ]:
import os
os.environ['GROQ_API_KEY']='*ENTER YOUR GROQ API KEY*'


In [ ]:
from google.colab import files
upload= files.upload()

Saving somatosensory.pdf to somatosensory (1).pdf
Saving iso27001.pdf to iso27001 (2).pdf
Saving drylab.pdf to drylab (3).pdf


In [ ]:
!pip install sentence-transformers

In [ ]:
pip uninstall -y langchain langchain-community openai

Found existing installation: langchain 1.3.6
Uninstalling langchain-1.3.6:
  Successfully uninstalled langchain-1.3.6
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: openai 2.41.0
Uninstalling openai-2.41.0:
  Successfully uninstalled openai-2.41.0


In [ ]:
pip install -q langchain langchain_community langchain_openai faiss-cpu pypdf sentence-transformers

In [ ]:
pip install -q langchain_groq

In [ ]:
from langchain_community.document_loaders import  PyPDFLoader
documents= []
for pdf_file in upload.keys():
    print("Loading:",pdf_file)
    loader= PyPDFLoader(pdf_file)
    documents.extend(loader.load())
print("Total pages loaded:",len(documents))

/tmp/ipykernel_20874/2671473631.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import  PyPDFLoader


Loading: somatosensory (1).pdf
Loading: iso27001 (2).pdf
Loading: drylab (3).pdf
Total pages loaded: 33


In [ ]:
print(documents[2].page_content)

Rapidly adapting Slowly adapting
Surface receptor /
small receptive
field
Hair receptor, Meissner’s corpuscle: De-
tect an insect or a very fine vibration.
Used for recognizing texture.
Merkel’s receptor: Used for spa-
tial details, e.g. a round surface
edge or “an X” in brail.
Deep receptor /
large receptive
field
Pacinian corpuscle: “A diffuse vibra-
tion” e.g. tapping with a pencil.
Ruffini’s corpuscle: “A skin
stretch”. Used for joint position
in fingers.
Table 1
Notice how figure captions and
sidenotes are shown in the outside
margin (on the left or right, depending
on whether the page is left or right).
Also, figures are floated to the top/
bottom of the page. Wide content, like
the table and Figure 3, intrude into the
outside margins.
only to intense mechanical stimuli, but also to heat and
to noxious chemicals. These rec eptors respond to minute
punctures of the epithelium, with a response magnitude
that depends on the degree of tissue deformation. They al-
so respond to temper

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(input("Enter chunk size (500/1000/1500): "))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print("Total chunks:", len(docs))

Enter chunk size (500/1000/1500): 1000
Total chunks: 93


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
print("Choose Embedding Model")
print("1.MibiLM")
print('2.MPNet')
print('3.BGE Small')
print('4. BGE base')

choice= input("Enter your choice: ")

models = {
    "1": "sentence-transformers/all-MiniLM-L6-v2",
    "2": "sentence-transformers/all-mpnet-base-v2",
    "3": "BAAI/bge-small-en-v1.5",
    "4": "BAAI/bge-base-en-v1.5"
}

model_name = models.get(choice)

print("Using:", model_name)

embeddings = HuggingFaceEmbeddings(model_name=model_name)

Choose Embedding Model
1.MibiLM
2.MPNet
3.BGE Small
4. BGE base
Enter your choice: 2
Using: sentence-transformers/all-mpnet-base-v2


/tmp/ipykernel_20874/1537646642.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstores= FAISS.from_documents(docs,embeddings)
print("Vector DB successfully created")
print(vectorstores)
vectorstores.save_local("faiss_index")

Vector DB successfully created


In [ ]:
k = int(input("How many chunks to retrieve? (3/5/8): "))

retriever = vectorstores.as_retriever(
    search_kwargs={"k": k}
)

How many chunks to retrieve? (3/5/8): 5


In [ ]:
print(vectorstores.index_to_docstore_id)

{0: '80dee7d2-31aa-431e-96f5-94044657604e', 1: 'e49a4ada-4562-4dc0-a95e-4cac0bf692ad', 2: '0128e9d4-b05c-4f7f-ae09-4ba57be51c0b', 3: '38b59098-cbe6-4947-b4df-ca78202273b6', 4: 'bbc4ab93-1d2b-4e80-9075-50debaad0ef8', 5: 'b7364a30-36a2-4eb3-98d6-00b05c60354a', 6: 'a471a0e3-7110-4caf-bd6a-d92c6ce657c3', 7: 'eec1fc26-89ff-40fd-bb65-98a4aa5e78cf', 8: '8d0131ad-cc6f-4c8c-88fb-4b454236dcd6', 9: '20599375-41de-4165-bb93-30f25e3fe8a0', 10: 'e8fc8aa2-231e-4bda-b222-8072faa68e44', 11: 'b5ab47a2-4f5b-4df4-8c14-b6a5c774970d', 12: '6ccde403-b8b2-48b1-b76f-f158f87e46b9', 13: '99072afd-dec4-44b4-83bb-a76abb48d9ff', 14: 'f14bf562-166e-4e36-805a-2dfcb6b24e85', 15: '2398cf17-109d-43d4-90f4-18feb02451fd', 16: '3e2d59a7-8c6c-40cc-a9b4-21af1e9e3992', 17: '8ab06303-b73e-4d4f-afdf-ae21c32ba014', 18: '3087d4ec-e86d-48ef-b92b-e81aecd009b8', 19: '4342fe6d-0148-458b-a08a-e436aa08ef6c', 20: 'a65c5e7d-31cf-491a-97f5-bf7e0d2ed1c9', 21: '6d168f39-fe4c-4d4d-a162-bee467a360cd', 22: '64cf8312-9893-4bc7-99c1-3671eefd1725

In [ ]:
docs_list= list(vectorstores.docstore._dict.values())
print(docs_list[0].page_content)

This is a sample document to
showcase page-based formatting. It
contains a chapter from a Wikibook
called Sensory Systems. None of the
content has been changed in this
article, but some content has been
removed.
Anatomy of the Somatosensory System
FROM WIKIBOOKS1
Our somatosensory system consists of sensors in the skin
and sensors in our muscles, tendons, and joints. The re-
ceptors in the skin, the so called cutaneous rec eptors, tell
us about temperature (thermoreceptors), pressure and sur-
face te xture ( mechano rec eptors), and pain ( nociceptors).
The receptors in muscles and joints pro vide information
about muscle length, muscle tension, and joint angles.
Cutaneous receptors
Sensory information from Meissner corpuscles and rapidly
adapting afferents leads to adjustment of grip f orce when
objects are lif ted. These aff erents respond with a brief
burst of action potentials when objects move a small dis-
tance during the early stages of lif ting. In response to


In [ ]:
for i in range(vectorstores.index.ntotal):
    vector = vectorstores.index.reconstruct(i)
    print(f"vector {i}: ")
    print(vector)
    print("-"*50)

Streaming output truncated to the last 5000 lines.
 -1.48713849e-02 -1.20076239e-02  9.79029294e-03  2.56502181e-02
  7.98174087e-03 -1.78356171e-02 -3.52631286e-02  7.11358804e-03
 -2.59992555e-02 -6.67085499e-02  4.59077349e-03  3.49658206e-02
  6.74202144e-02 -5.64667676e-03 -8.28943178e-02 -8.57464410e-03
 -2.40722839e-02 -1.38338283e-02 -4.35110405e-02 -1.91391539e-02
 -6.63458332e-02  1.61179584e-02 -1.98240634e-02  5.32059669e-02
  9.32738092e-03 -2.29125507e-02 -4.46843989e-02  4.83401772e-03
 -1.82443261e-02 -1.21443132e-02  1.51145579e-02  8.95768553e-02
 -1.82214286e-02  3.17136124e-02 -6.91677153e-04 -5.24853133e-02
 -1.22409416e-02  2.07453184e-02  2.61272918e-02 -6.39153784e-03
  3.40325832e-02  1.21796140e-02 -2.11995337e-02  5.62221259e-02
  1.61259919e-02  1.77531724e-03 -5.51144816e-02  3.11723799e-02
 -1.59331430e-02  5.67987375e-02 -4.73015606e-02  3.17755528e-02
  1.53080868e-02 -2.61677839e-02 -2.86065228e-02  2.16345172e-02
 -5.38497092e-03  4.35131341e-02  2.916

In [ ]:
from langchain_openai import ChatOpenAI
import os
llm= ChatOpenAI(model="llama-3.3-70b-versatile",
                api_key=os.getenv('GROQ_API_KEY'),
                base_url="http://api.groq.com/openai/v1")

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template(
    """ Answer the question base only on the context below:
    {context}
    Question: {question}
    """
)


# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


query = input("Ask a question: ")

docs=retriever.invoke(query)
print('Retreived docs:',len(docs))

ch = input("Enable reranking? (y/n): ").strip().lower()

if ch == "y":
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)

    docs = [
        doc for _, doc in sorted(
            zip(scores, docs),
            key=lambda x: x[0],
            reverse=True
        )
    ]

    print("✅ Reranking Enabled")
    print("\nReranking Scores:")

    for i, (score, doc) in enumerate(
        sorted(zip(scores, docs),
               key=lambda x: x[0],
               reverse=True),
        start=1
    ):
        print(f"{i}. Score: {score:.4f}")

else:
    print("✅ Using FAISS Retrieval")


context = format_docs(docs)

result = (
    prompt
    | llm
    | StrOutputParser()
).invoke({
    "context": context,
    "question": query
})

print("\n Answer:\n")
print(result)

Ask a question: Cutaneous receptors
Retreived docs: 5
Enable reranking? (y/n): n
✅ Using FAISS Retrieval

 Answer:

Cutaneous receptors in the skin provide information about:

1. Temperature (thermoreceptors)
2. Pressure and surface texture (mechano receptors)
3. Pain (nociceptors)

They can be classified into two main categories:

1. Rapidly adapting receptors: 
   - Respond to initial indentation of skin
   - Examples: Meissner's corpuscle, Pacinian corpuscle, Hair receptor

2. Slowly adapting receptors: 
   - Respond to sustained indentation of skin
   - Examples: Merkel's receptor, Ruffini's corpuscle

These receptors are found in different densities in various parts of the body, such as:

- High density in digits and around the mouth (50/mm² of skin surface)
- Lower density in other glabrous surfaces
- Very low density in hairy skin

The density of these receptors decreases with age, with a significant reduction by the age of 50.
